In [ ]:
import rospy
from geometry_msgs.msg import Point, Pose, Twist
from nav_msgs.msg import Odometry
from sensor_msgs.msg import LaserScan 
import actionlib
import actionlib.msg
import assignment_2_2024.msg
from assignment_2_2024.msg import PositionVelocity
from assignment_2_2024.msg import PlanningAction, PlanningGoal, PlanningResult, PlanningActionResult
from std_srvs.srv import SetBool
from actionlib_msgs.msg import GoalStatus

import ipywidgets as widgets
from ipywidgets import FloatText, Button, VBox, Label, HBox, Output
from IPython.display import display

In [ ]:
# ---- Initialize ROS Node ----
rospy.init_node('notebook_client', anonymous=True)

# Action client
client = actionlib.SimpleActionClient('/reaching_goal', PlanningAction)
client.wait_for_server()

In [ ]:
# Global variables to store robot info
current_pos = {"x": 0.0, "y": 0.0}
current_vel = {"vx": 0.0, "vz": 0.0}
closest_obstacle = None

In [ ]:
# ---- Widgets ----
x_input = FloatText(description='Target X:', value=0.0)
y_input = FloatText(description='Target Y:', value=0.0)
set_goal_btn = Button(description='Set Goal', button_style='success')
cancel_goal_btn = Button(description='Cancel Goal', button_style='danger')
status_output = Output()

robot_pos_label = Label("Current Position: N/A")
robot_vel_label = Label("Current Velocity: N/A")
obstacle_label = Label("Closest Obstacle: N/A")

In [ ]:
# ---- Callbacks ----
def odom_callback(msg):
    current_pos['x'] = msg.pose.pose.position.x
    current_pos['y'] = msg.pose.pose.position.y
    current_vel['vx'] = msg.twist.twist.linear.x
    current_vel['vz'] = msg.twist.twist.angular.z

    robot_pos_label.value = f"Current Position: ({current_pos['x']:.2f}, {current_pos['y']:.2f})"
    robot_vel_label.value = f"Velocity - Linear: {current_vel['vx']:.2f}, Angular: {current_vel['vz']:.2f}"

def scan_callback(msg):
    ranges = [r for r in msg.ranges if r > 0.01]
    if ranges:
        closest = min(ranges)
        obstacle_label.value = f"Closest Obstacle: {closest:.2f} m"
    else:
        obstacle_label.value = "Closest Obstacle: N/A"

In [ ]:
# ---- Action Functions ----
def set_goal_callback(btn):
    goal = PlanningGoal()
    goal.target_pose.pose.position.x = x_input.value
    goal.target_pose.pose.position.y = y_input.value
    client.send_goal(goal)

    with status_output:
        print(f"Goal sent: ({x_input.value}, {y_input.value})")

def cancel_goal_callback(btn):
    client.cancel_goal()
    with status_output:
        print("Goal cancelled.")

In [ ]:
# ---- Widget Events ----
set_goal_btn.on_click(set_goal_callback)
cancel_goal_btn.on_click(cancel_goal_callback)

# ---- ROS Subscribers ----
rospy.Subscriber("/odom", Odometry, odom_callback)
rospy.Subscriber("/scan", LaserScan, scan_callback)  # Optional: if /scan available

In [ ]:
# ---- Display UI ----
ui = VBox([
    HBox([x_input, y_input]),
    HBox([set_goal_btn, cancel_goal_btn]),
    robot_pos_label,
    robot_vel_label,
    obstacle_label,
    status_output
])

display(ui)